# OpsLM — QLoRA fine-tune of Qwen3-4B (Colab T4)

Turnkey training run for **OpsLM**. Runtime -> **Change runtime type -> T4 GPU**.

This notebook is a thin wrapper over `training/scripts/train_opslm_qlora.py`
in the repo (the script is the source of truth; the notebook just drives it).
You need: a **Hugging Face token** with write access. Nothing else.

Flow: install -> clone repo -> get the SFT data -> train (resumable) ->
the script pushes merged 16-bit + GGUF Q4_K_M to your HF model repo.

## 1. Set your names + token

In [ ]:
import os
from getpass import getpass

HF_USER = "your-hf-username"           # <-- edit
PUSH_REPO = f"{HF_USER}/OpsLM-v1"      # the model repo to create/push
os.environ["HF_TOKEN"] = getpass("HF token (write access): ")
print("will push to:", PUSH_REPO)

## 2. Install pinned deps

In [ ]:
!pip -q install "unsloth==2025.*" "trl>=0.12,<0.20" "datasets>=3" "huggingface_hub>=0.25"

## 3. Get the code and the SFT data

The 534/59 train/val split (produced locally by
`training/scripts/prepare_sft.py`) is **committed to the repo** at
`data/sft/{train,val}.jsonl` — the clone below already contains it.
No upload needed; the assert just confirms it.


In [ ]:
!git clone --depth 1 https://github.com/dilipna/OPsVerse.git
%cd OPsVerse

import os
assert os.path.exists("data/sft/train.jsonl"), "data/sft missing from clone - check the repo"
print(sum(1 for _ in open("data/sft/train.jsonl")), "train examples")


## 4. Train (resumable)

~3–4h on a T4 for 3 epochs. If the session dies, re-run this cell with
`--resume` appended — checkpoints are pushed to the Hub every 50 steps.

In [ ]:
!python training/scripts/train_opslm_qlora.py \
    --data-repo data/sft \
    --push-repo {PUSH_REPO} \
    --epochs 3 --save-steps 50

## 5. After training — the before/after eval

Back on the OpsVerse stack (not Colab), serve OpsLM (e.g. Ollama from the
pushed GGUF) and run the Phase-4 harness against base Qwen3-4B vs OpsLM-v1:

```bash
# point chat_model at the served OpsLM, then:
uv run python -m opsverse_evals.rag_suite --n 20
```

That produces the honest before/after report the resume line stands on.
The eval gate that OpsLM must clear already exists — that ordering is the
point.